# Tutorial 10: Annotation and Entity Context

Annotation utilities collect biological context. Whenever that context needs
to become an embedding, this tutorial uses `BioEmbedder.embed(...)`.


In [ ]:
from embpy import BioEmbedder
from embpy.resources import GeneAnnotator, MoleculeAnnotator
from embpy.resources.text import TextResolver

RUN_EMBEDDING = False
embedder = BioEmbedder(device="auto", organism="human")
gene_annotator = GeneAnnotator(organism="human")
molecule_annotator = MoleculeAnnotator()
text_resolver = TextResolver(organism="human")


## 1. Gene annotations as text embeddings


In [ ]:
genes = ["TP53", "BRCA1", "EGFR"]
gene_contexts = []
for gene in genes:
    ann = gene_annotator.annotate(gene)
    pathways = ", ".join(ann.get("pathways", [])[:5]) or "no pathways found"
    diseases = ", ".join(d.get("disease", "") for d in ann.get("diseases", [])[:5])
    gene_contexts.append(f"{gene}. Pathways: {pathways}. Disease links: {diseases}.")

if RUN_EMBEDDING:
    gene_text = embedder.embed(
        gene_contexts,
        entity_type="text",
        model="minilm_l6_v2",
        output="anndata",
        key="X_gene_context_minilm",
    )
    gene_text.obs["gene"] = genes
    print(gene_text.obsm["X_gene_context_minilm"].shape)


## 2. Molecule annotations as text embeddings


In [ ]:
drugs = ["aspirin", "caffeine", "ibuprofen"]
drug_contexts = []
for drug in drugs:
    ann = molecule_annotator.annotate(drug)
    moa = ann.get("mechanism_of_action") or "mechanism unavailable"
    targets = ", ".join(ann.get("targets", [])[:5]) or "targets unavailable"
    drug_contexts.append(f"{drug}. Mechanism: {moa}. Targets: {targets}.")

if RUN_EMBEDDING:
    drug_text = embedder.embed(
        drug_contexts,
        entity_type="text",
        model="minilm_l6_v2",
        output="payload",
        key="X_drug_context_minilm",
    )
    print(drug_text["n_entities"], drug_text["n_dims"])


## 3. Resolver descriptions are also text inputs


In [ ]:
descriptions = [text_resolver.describe(gene, entity_type="gene") for gene in genes]

if RUN_EMBEDDING:
    desc_table = embedder.embed(
        descriptions,
        entity_type="text",
        model="minilm_l6_v2",
        output="table",
        path="gene_description_embeddings.csv",
        fmt="csv",
    )
    print(desc_table.head())
